# nb02 - SQL and pandas parity: Medi-Cal market share in Los Angeles

Every figure on the Tableau dashboard **The Largest Medi-Cal Plan in California, and the Access Gap Around It** is reproduced here twice, once in DuckDB SQL and once in pandas, and asserted against the number Tableau actually renders. If an assert fails, the blog post is wrong.

Published dashboard: https://public.tableau.com/views/la_medi_cal_market_share/Dashboard1

**The point of this notebook is the translation, not the numbers.** Each Tableau feature has an exact SQL and pandas analog:

| Tableau | SQL | pandas |
|---|---|---|
| `{ FIXED [Month] : SUM([Enrollees]) }` | `SUM(...) OVER (PARTITION BY Month)` | `groupby('Month')[...].transform('sum')` |
| A join on County (penetration, access) | `JOIN ... ON County` | `merge(..., on='County')` |
| A join on Brand and Year (quality) | `JOIN ... ON Brand AND Year` | `merge(..., on=['Brand','Year'])` |
| Add to Context (the size filter) | move the condition into the **CTE**, before the window | filter the frame **before** the groupby |

**Data note.** The three cleaning steps live in `nb01` (era normalization of the two Plan Type labels, grouping the plan name variants, filtering out the six non-medical product lines). The two enrichment reference files (`ca_county_population.csv`, `la_plan_quality.csv`) are written by `nb01b`, and the provider files by `nb01c`. This notebook only reads the clean outputs, so the joins here are the same joins performed in Tableau.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA = Path('..') / 'data'
share   = pd.read_csv(DATA / 'la_market_share_clean.csv')      # month x brand, LA County
county  = pd.read_csv(DATA / 'ca_county_enrollment_clean.csv') # latest month, 58 counties
pop     = pd.read_csv(DATA / 'ca_county_population.csv')        # CA DOF E-2 2024
prov    = pd.read_csv(DATA / 'providers_by_county.csv')         # distinct NPI per county
quality = pd.read_csv(DATA / 'la_plan_quality.csv')             # HEDIS AQFS, LA region
ptype   = pd.read_csv(DATA / 'la_care_providers_by_type.csv')   # LA Care network by type

con = duckdb.connect()
for name, df in [('share', share), ('county', county), ('pop', pop),
                 ('prov', prov), ('quality', quality), ('ptype', ptype)]:
    con.register(name, df)

MONTH = '2026-06'   # latest enrollment month in the series

for n, df in [('share', share), ('county', county), ('prov', prov)]:
    print(f'{n:8s}', df.shape)

RESULTS = []
def check(name, tableau, sql, pd_, places=3):
    """Assert the SQL and pandas answers both match what Tableau renders."""
    ok = (round(float(sql), places) == round(float(tableau), places)
          and round(float(pd_), places) == round(float(tableau), places))
    RESULTS.append((name, tableau, round(float(sql), places), round(float(pd_), places), ok))
    flag = 'PASS' if ok else 'FAIL'
    print(f'[{flag}] {name}: tableau={tableau} sql={round(float(sql), places)} pandas={round(float(pd_), places)}')
    assert ok, name

share    (498, 6)
county   (58, 3)
prov     (58, 3)


## 1. `Market Share` - the FIXED LOD

Tableau: `Month Total = { FIXED [Enrollment Month] : SUM([Enrollees]) }`, then `Market Share = SUM([Enrollees]) / SUM([Month Total])`. The FIXED total is computed per month before the brand breakdown, so each brand's share divides by the whole month, not by itself.

- **SQL:** `SUM(Enrollees) OVER (PARTITION BY month)` is the window analog of the FIXED total; the share is the row value divided by that window total.
- **pandas:** `groupby('Enrollment Month').transform('sum')` broadcasts the month total onto every brand row.

For June 2026: L.A. Care 59.7%, Health Net 30.7%, Kaiser 9.6%.

In [2]:
sql = con.execute("""
    WITH totals AS (                                         -- the FIXED [Month] total
        SELECT "Enrollment Month" AS month, Brand, Enrollees,
               SUM(Enrollees) OVER (PARTITION BY "Enrollment Month") AS month_total
        FROM share
    )
    SELECT Brand, Enrollees / month_total AS market_share
    FROM totals
    WHERE month = ?
    ORDER BY market_share DESC
""", [MONTH]).df()

s = share.copy()
s['month_total'] = s.groupby('Enrollment Month')['Enrollees'].transform('sum')
s['market_share'] = s['Enrollees'] / s['month_total']
pdf = s.loc[s['Enrollment Month'] == MONTH, ['Brand', 'market_share']]

print(sql.to_string(index=False))

for brand, tab in [('L.A. Care', 0.597), ('Health Net', 0.3067), ('Kaiser Permanente', 0.0963)]:
    check(f'Market Share {brand}', tab,
          sql.loc[sql.Brand == brand, 'market_share'].iloc[0],
          pdf.loc[pdf.Brand == brand, 'market_share'].iloc[0], places=4)

            Brand  market_share
        L.A. Care      0.597030
       Health Net      0.306666
Kaiser Permanente      0.096304
[PASS] Market Share L.A. Care: tableau=0.597 sql=0.597 pandas=0.597
[PASS] Market Share Health Net: tableau=0.3067 sql=0.3067 pandas=0.3067
[PASS] Market Share Kaiser Permanente: tableau=0.0963 sql=0.0963 pandas=0.0963


## 2. `Enrollment Over Time` - the 19 year market total

Tableau: a line of `SUM([Enrollees])` per `[Enrollment Month]`, which is the same monthly total as the FIXED denominator above. The story markers: the market opens near 1.18M (Jan 2007), roughly doubles after the 2014 ACA Medicaid expansion, peaks at 3.95M in June 2023, then falls to 3.52M by June 2026 as the post COVID eligibility redetermination unwinds.

- **SQL:** `SUM(Enrollees) GROUP BY month`.
- **pandas:** `groupby('Enrollment Month')['Enrollees'].sum()`.

In [3]:
sql = con.execute("""
    SELECT "Enrollment Month" AS month, SUM(Enrollees) AS total
    FROM share GROUP BY month ORDER BY month
""").df()

pdf = share.groupby('Enrollment Month')['Enrollees'].sum()

def m(month): return sql.loc[sql.month == month, 'total'].iloc[0]
print('peak:', sql.loc[sql.total.idxmax(), 'month'], int(sql.total.max()))

for month, tab in [('2007-01', 1177988), ('2014-01', 2029015),
                   ('2023-06', 3950362), ('2026-06', 3521674)]:
    check(f'Enrollment {month}', tab, m(month), pdf[month], places=0)

peak: 2023-06 3950362
[PASS] Enrollment 2007-01: tableau=1177988 sql=1177988.0 pandas=1177988.0
[PASS] Enrollment 2014-01: tableau=2029015 sql=2029015.0 pandas=2029015.0
[PASS] Enrollment 2023-06: tableau=3950362 sql=3950362.0 pandas=3950362.0
[PASS] Enrollment 2026-06: tableau=3521674 sql=3521674.0 pandas=3521674.0


## 3. `Penetration Map` - a join on County

Tableau performs a relationship between the county enrollment extract and the county population reference file on `County`, then `Penetration = SUM([Medi-Cal MC Enrollees]) / SUM([Population])`. Los Angeles sits at 36.4%; the highest reliance is inland (Tulare 55.1%, Imperial 52.2%).

- **SQL:** `JOIN pop ON county.County = pop.County`.
- **pandas:** `county.merge(pop, on='County')`.

In [4]:
sql = con.execute("""
    SELECT c.County,
           c."Medi-Cal MC Enrollees" / p.Population AS penetration
    FROM county c JOIN pop p ON c.County = p.County
""").df()

pdf = county.merge(pop, on='County')
pdf['penetration'] = pdf['Medi-Cal MC Enrollees'] / pdf['Population']

def pen(df, c, col): return df.loc[df.County == c, col].iloc[0] * 100
for c, tab in [('Los Angeles', 36.4), ('Tulare', 55.1), ('Imperial', 52.2)]:
    check(f'Penetration {c}', tab,
          sql.loc[sql.County == c, 'penetration'].iloc[0] * 100,
          pdf.loc[pdf.County == c, 'penetration'].iloc[0] * 100, places=1)

[PASS] Penetration Los Angeles: tableau=36.4 sql=36.4 pandas=36.4
[PASS] Penetration Tulare: tableau=55.1 sql=55.1 pandas=55.1
[PASS] Penetration Imperial: tableau=52.2 sql=52.2 pandas=52.2


## 4. `Access Map` - a join on County, plus a Context filter

Tableau joins the same county enrollment to the provider counts on `County`, then `Providers per 1,000 Members = SUM([Providers All]) / SUM([Medi-Cal MC Enrollees]) * 1000`. A `Medi-Cal MC Enrollees >= 5,000` filter is set to **Add to Context** so tiny denominator counties (Alpine, Sierra) cannot distort the color scale. Add to Context is the exact analog of pushing the condition into a CTE so it runs before the ratio is formed.

Los Angeles has 13.1 providers per 1,000 members; the thinnest networks are exactly the counties that lean hardest on Medi-Cal (Imperial 2.9, Tulare 6.7) - the coverage vs access gap.

- **SQL:** the `WHERE ... >= 5000` lives in the CTE (context), the ratio is computed outside it.
- **pandas:** filter the frame before forming the ratio.

In [5]:
sql = con.execute("""
    WITH in_context AS (                                     -- Add to Context: filter first
        SELECT c.County, c."Medi-Cal MC Enrollees" AS members, pr.Providers_All AS providers
        FROM county c JOIN prov pr ON c.County = pr.County
        WHERE c."Medi-Cal MC Enrollees" >= 5000
    )
    SELECT County, providers / members * 1000 AS per_1k
    FROM in_context ORDER BY per_1k
""").df()

pdf = county.merge(prov, on='County')
pdf = pdf[pdf['Medi-Cal MC Enrollees'] >= 5000].copy()
pdf['per_1k'] = pdf['Providers_All'] / pdf['Medi-Cal MC Enrollees'] * 1000

for c, tab in [('Los Angeles', 13.1), ('Imperial', 2.9), ('Tulare', 6.7)]:
    check(f'Providers per 1k {c}', tab,
          sql.loc[sql.County == c, 'per_1k'].iloc[0],
          pdf.loc[pdf.County == c, 'per_1k'].iloc[0], places=1)

[PASS] Providers per 1k Los Angeles: tableau=13.1 sql=13.1 pandas=13.1
[PASS] Providers per 1k Imperial: tableau=2.9 sql=2.9 pandas=2.9
[PASS] Providers per 1k Tulare: tableau=6.7 sql=6.7 pandas=6.7


## 5. `Share vs Quality` - a join on Brand and Year

Tableau joins the LA market share to the HEDIS quality reference (`la_plan_quality.csv`, the Aggregated Quality Factor Score for the Los Angeles region) on both `Brand` and `Year`. Only L.A. Care and Health Net have a Medi-Cal quality score in LA; Kaiser's LA Medi-Cal line launched in 2024 and has no reported score, so it does not appear on the scatter. For 2023: L.A. Care 59.33, Health Net 52.00.

- **SQL:** `JOIN quality ON Brand AND Year`.
- **pandas:** `merge(quality, on=['Brand','Year'])`.

In [6]:
sql = con.execute("""
    SELECT Brand, AQFS FROM quality WHERE Year = 2023 ORDER BY AQFS DESC
""").df()

pdf = quality[quality.Year == 2023]

for brand, tab in [('L.A. Care', 59.33), ('Health Net', 52.00)]:
    check(f'AQFS 2023 {brand}', tab,
          sql.loc[sql.Brand == brand, 'AQFS'].iloc[0],
          pdf.loc[pdf.Brand == brand, 'AQFS'].iloc[0], places=2)

[PASS] AQFS 2023 L.A. Care: tableau=59.33 sql=59.33 pandas=59.33
[PASS] AQFS 2023 Health Net: tableau=52.0 sql=52.0 pandas=52.0


## 6. `L.A. Care Providers by Type` - the network composition

Tableau: a bar of `SUM([Providers])` per `[Provider Type]`, with the catch all `Other` bucket excluded. The network runs on primary care: Adult Primary Care (1,125), Nurse Practitioner (1,067), and Pediatric Primary Care (938) are the three largest specialties.

- **SQL:** `SUM(Providers) GROUP BY "Provider Type"`.
- **pandas:** `groupby('Provider Type')['Providers'].sum()`.

In [7]:
sql = con.execute("""
    SELECT "Provider Type" AS ptype, SUM(Providers) AS n
    FROM ptype WHERE "Provider Type" <> 'Other'
    GROUP BY "Provider Type" ORDER BY n DESC
""").df()

pdf = ptype[ptype['Provider Type'] != 'Other'].groupby('Provider Type')['Providers'].sum()

for t, tab in [('Adult Primary Care', 1125), ('Nurse Practitioner', 1067),
               ('Pediatric Primary Care', 938)]:
    check(f'Providers {t}', tab,
          sql.loc[sql.ptype == t, 'n'].iloc[0], pdf[t], places=0)

[PASS] Providers Adult Primary Care: tableau=1125 sql=1125.0 pandas=1125.0
[PASS] Providers Nurse Practitioner: tableau=1067 sql=1067.0 pandas=1067.0
[PASS] Providers Pediatric Primary Care: tableau=938 sql=938.0 pandas=938.0


## Parity summary

In [8]:
out = pd.DataFrame(RESULTS, columns=['figure', 'tableau', 'duckdb_sql', 'pandas', 'match'])
print(out.to_string(index=False))
print()
print(f'{out.match.sum()} of {len(out)} checks pass')
assert out.match.all(), 'a figure on the dashboard does not reconcile'
print('ALL PARITY CHECKS PASS - every published figure reconciles in Tableau, DuckDB SQL, and pandas')

                          figure      tableau   duckdb_sql       pandas  match
          Market Share L.A. Care       0.5970       0.5970       0.5970   True
         Market Share Health Net       0.3067       0.3067       0.3067   True
  Market Share Kaiser Permanente       0.0963       0.0963       0.0963   True
              Enrollment 2007-01 1177988.0000 1177988.0000 1177988.0000   True
              Enrollment 2014-01 2029015.0000 2029015.0000 2029015.0000   True
              Enrollment 2023-06 3950362.0000 3950362.0000 3950362.0000   True
              Enrollment 2026-06 3521674.0000 3521674.0000 3521674.0000   True
         Penetration Los Angeles      36.4000      36.4000      36.4000   True
              Penetration Tulare      55.1000      55.1000      55.1000   True
            Penetration Imperial      52.2000      52.2000      52.2000   True
    Providers per 1k Los Angeles      13.1000      13.1000      13.1000   True
       Providers per 1k Imperial       2.9000       